# 31｜不用预制 Transformer：手写 Decoder-only GPT 与 KV Cache

本笔记从 token/位置嵌入、手写 QKV 多头注意力、Pre-Norm 残差块一直实现到 `ScratchGPT.forward` 和逐 token `forward_step`。重点不是堆出一个“大模型”，而是把三条容易写错的工程合同变成可执行证据：**未来 token 不得泄漏、全量前向与 KV Cache 的 logits 必须等价、tokenizer/config/权重必须被同一制品指纹绑定**。

> 实验边界：固定 CPU、合成小数据和受控过拟合只能验证实现链路，不能代表真实语言建模能力。

## 1. 输入、输出与验收合同

- `input_ids: [B,T]`，`valid_mask: [B,T]`；有效 token 必须是左对齐连续前缀，且每条至少一个。
- 隐状态为 `[B,T,D]`；拆头后为 `[B,H,T,D/H]`，要求 `D % H == 0`。
- full forward 同时使用 causal mask 与 key padding mask；padding query 的输出强制归零。
- 增量接口一次只接收 `[B,1]`，第 `l` 层 cache 保存 `K/V: [B,H,t,d_h]` 和 key 有效位。
- LM head 与 token embedding **共享同一 Parameter**，而不只是数值拷贝。
- 所有随机源固定；不联网、不下载数据、不依赖 `nn.MultiheadAttention/nn.Transformer`。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 20260810  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
PAD, BOS, EOS, UNK = 0, 1, 2, 3  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert len({PAD, BOS, EOS, UNK}) == 4  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. 先冻结 tokenizer 合同

生产中的模型权重依赖“字符串如何变成 id”。只校验 `state_dict` 而漏掉词表顺序，会让同一 id 悄悄换义。下面的离线 tokenizer 故意很小：按空格切分、可添加 BOS/EOS，并把 token 顺序纳入制品哈希。它不是通用分词器，只用于展示接口和版本边界。

In [ ]:
class TinyTokenizer:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, tokens):  # 定义本节可复用的核心函数。
        if len(tokens) != len(set(tokens)):  # 按当前条件选择后续控制路径。
            raise ValueError("词表 token 必须唯一")  # 遇到非法合同立即显式失败。
        if list(tokens[:4]) != ["<pad>", "<bos>", "<eos>", "<unk>"]:  # 按当前条件选择后续控制路径。
            raise ValueError("特殊 token 的 id 合同不成立")  # 遇到非法合同立即显式失败。
        self.tokens = list(tokens)  # 计算并保存当前步骤的中间状态。
        self.to_id = {token: i for i, token in enumerate(tokens)}  # 计算并保存当前步骤的中间状态。

    def encode(self, text, add_special=True):  # 定义本节可复用的核心函数。
        ids = [self.to_id.get(piece, UNK) for piece in text.strip().split()]  # 计算并保存当前步骤的中间状态。
        return ([BOS] + ids + [EOS]) if add_special else ids  # 返回当前分支计算出的结果。

    def decode(self, ids, skip_special=True):  # 定义本节可复用的核心函数。
        specials = {PAD, BOS, EOS}  # 计算并保存当前步骤的中间状态。
        return " ".join(self.tokens[i] for i in ids if not (skip_special and i in specials))  # 返回当前分支计算出的结果。

    def spec(self):  # 定义本节可复用的核心函数。
        return {"kind": "whitespace-v1", "tokens": self.tokens}  # 返回当前分支计算出的结果。

TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "我", "爱", "机器", "学习",  # 计算并保存当前步骤的中间状态。
          "模型", "需要", "数据", "检索", "图", "视觉", "文本", "安全"]  # 执行当前语句以推进本节示例。
tokenizer = TinyTokenizer(TOKENS)  # 计算并保存当前步骤的中间状态。
roundtrip = tokenizer.encode("我 爱 机器 学习")  # 计算并保存当前步骤的中间状态。
assert roundtrip == [BOS, 4, 5, 6, 7, EOS]  # 用受控断言验证关键不变量。
assert tokenizer.decode(roundtrip) == "我 爱 机器 学习"  # 用受控断言验证关键不变量。
assert tokenizer.encode("未知词", add_special=False) == [UNK]  # 用受控断言验证关键不变量。

## 3. 缩放点积注意力与 mask 语义

对第 `h` 个头：

$$A_h=\operatorname{softmax}\left(\frac{Q_hK_h^\top}{\sqrt{d_h}}+M\right),\qquad O_h=A_hV_h.$$

`allowed_mask=True` 表示可读。full forward 的允许矩阵是

$$M_{ij}=\mathbb{1}[j\le i]\land\mathbb{1}[\text{key}_j\text{ valid}].$$

代码拒绝整行无可见 key，避免 softmax 全是负无穷时产生 NaN。padding query 虽可计算临时值，但返回前会乘 query mask 清零。

In [ ]:
def scaled_dot_product_attention(q, k, v, allowed_mask):  # 定义本节可复用的核心函数。
    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4:  # 按当前条件选择后续控制路径。
        raise ValueError("q/k/v 必须是 [B,H,T,d_h]")  # 遇到非法合同立即显式失败。
    if k.shape != v.shape or q.shape[:2] != k.shape[:2] or q.shape[-1] != k.shape[-1]:  # 按当前条件选择后续控制路径。
        raise ValueError("q/k/v 形状合同不成立")  # 遇到非法合同立即显式失败。
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])  # 计算并保存当前步骤的中间状态。
    allowed = torch.broadcast_to(allowed_mask, scores.shape)  # 计算并保存当前步骤的中间状态。
    if allowed.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise TypeError("allowed_mask 必须为 bool")  # 遇到非法合同立即显式失败。
    if bool((~allowed.any(-1)).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("存在看不到任何 key 的 query")  # 遇到非法合同立即显式失败。
    scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)  # 计算并保存当前步骤的中间状态。
    weights = torch.softmax(scores, dim=-1)  # 计算并保存当前步骤的中间状态。
    return weights @ v, weights  # 返回当前分支计算出的结果。

q = torch.tensor([[[[1.0, 0.0]]]])  # 计算并保存当前步骤的中间状态。
k = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])  # 计算并保存当前步骤的中间状态。
v = torch.tensor([[[[3.0, 1.0], [9.0, 7.0]]]])  # 计算并保存当前步骤的中间状态。
context, weights = scaled_dot_product_attention(q, k, v, torch.tensor([[[[True, False]]]]))  # 计算并保存当前步骤的中间状态。
assert torch.allclose(context, v[:, :, :1])  # 用受控断言验证关键不变量。
assert torch.allclose(weights.sum(-1), torch.ones_like(weights.sum(-1)))  # 用受控断言验证关键不变量。

## 4. 手写 MHA、Pre-Norm block 与 GPT

每层先归一化再进入子层：

$$x'=x+\operatorname{MHA}(\operatorname{LN}(x)),\quad
x''=x'+\operatorname{MLP}(\operatorname{LN}(x')).$$

Pre-Norm 在深层网络中通常更容易优化。MLP 使用 $D\to4D\to D$。增量模式只为新 token 计算 Q/K/V，并将 K/V 追加到各层 cache；历史 token 的 K/V 不再重复投影。这里不用 dropout，使 train/eval 与 full/cache oracle 更容易精确比较。

In [ ]:
def validate_prefix_mask(mask):  # 定义本节可复用的核心函数。
    if mask.ndim != 2 or mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("valid_mask 必须是二维 bool")  # 遇到非法合同立即显式失败。
    if bool((~mask.any(1)).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("每条序列至少需要一个有效 token")  # 遇到非法合同立即显式失败。
    if mask.shape[1] > 1 and bool((mask[:, 1:] & ~mask[:, :-1]).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("有效 token 必须是左对齐连续前缀")  # 遇到非法合同立即显式失败。


class ManualCausalSelfAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, n_heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if d_model <= 0 or n_heads <= 0 or d_model % n_heads:  # 按当前条件选择后续控制路径。
            raise ValueError("d_model 必须能被 n_heads 整除")  # 遇到非法合同立即显式失败。
        self.d_model, self.n_heads = d_model, n_heads  # 计算并保存当前步骤的中间状态。
        self.head_dim = d_model // n_heads  # 计算并保存当前步骤的中间状态。
        self.q_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.k_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.v_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.out_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。

    def _split(self, x):  # 定义本节可复用的核心函数。
        B, T, _ = x.shape  # 计算并保存当前步骤的中间状态。
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # 返回当前分支计算出的结果。

    def _merge(self, x):  # 定义本节可复用的核心函数。
        B, H, T, Dh = x.shape  # 计算并保存当前步骤的中间状态。
        return x.transpose(1, 2).contiguous().view(B, T, H * Dh)  # 返回当前分支计算出的结果。

    def forward(self, x, valid_mask):  # 定义本节可复用的核心函数。
        B, T, D = x.shape  # 计算并保存当前步骤的中间状态。
        if D != self.d_model or valid_mask.shape != (B, T):  # 按当前条件选择后续控制路径。
            raise ValueError("attention 输入形状错误")  # 遇到非法合同立即显式失败。
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))  # 计算并保存当前步骤的中间状态。
        causal = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()  # 计算并保存当前步骤的中间状态。
        allowed = causal[None, None] & valid_mask[:, None, None, :]  # 计算并保存当前步骤的中间状态。
        context, weights = scaled_dot_product_attention(q, k, v, allowed)  # 计算并保存当前步骤的中间状态。
        y = self.out_proj(self._merge(context))  # 计算并保存当前步骤的中间状态。
        return y * valid_mask.unsqueeze(-1), weights  # 返回当前分支计算出的结果。

    def _validate_cache(self, cache, batch_size, expected_length, reference):  # 定义本节可复用的核心函数。
        required = {"k", "v", "valid"}  # 计算并保存当前步骤的中间状态。
        if not isinstance(cache, dict) or set(cache) != required:  # 按当前条件选择后续控制路径。
            raise ValueError("cache 字段必须严格为 k/v/valid")  # 遇到非法合同立即显式失败。
        k_old, v_old, valid_old = cache["k"], cache["v"], cache["valid"]  # 计算并保存当前步骤的中间状态。
        if not all(isinstance(value, torch.Tensor) for value in (k_old, v_old, valid_old)):  # 按当前条件选择后续控制路径。
            raise ValueError("cache 字段必须是 tensor")  # 遇到非法合同立即显式失败。
        expected_kv = (batch_size, self.n_heads, expected_length, self.head_dim)  # 计算并保存当前步骤的中间状态。
        if k_old.shape != expected_kv or v_old.shape != expected_kv:  # 按当前条件选择后续控制路径。
            raise ValueError(f"cache K/V shape 错误，期望 {expected_kv}")  # 遇到非法合同立即显式失败。
        if valid_old.shape != (batch_size, expected_length) or valid_old.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("cache valid shape/dtype 错误")  # 遇到非法合同立即显式失败。
        if k_old.dtype != reference.dtype or v_old.dtype != reference.dtype:  # 按当前条件选择后续控制路径。
            raise ValueError("cache K/V dtype 与当前请求不一致")  # 遇到非法合同立即显式失败。
        if k_old.device != reference.device or v_old.device != reference.device or valid_old.device != reference.device:  # 按当前条件选择后续控制路径。
            raise ValueError("cache device 与当前请求不一致")  # 遇到非法合同立即显式失败。
        validate_prefix_mask(valid_old)  # 执行当前语句以推进本节示例。
        return k_old, v_old, valid_old  # 返回当前分支计算出的结果。

    def forward_step(self, x_t, cache, valid_t, expected_length):  # 定义本节可复用的核心函数。
        if x_t.ndim != 3 or x_t.shape[1] != 1 or x_t.shape[-1] != self.d_model:  # 按当前条件选择后续控制路径。
            raise ValueError("增量 attention 一次只接收 [B,1,D]")  # 遇到非法合同立即显式失败。
        if valid_t.shape != x_t.shape[:2] or valid_t.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("valid_t 形状或类型错误")  # 遇到非法合同立即显式失败。
        if not isinstance(expected_length, int) or isinstance(expected_length, bool) or expected_length < 0:  # 按当前条件选择后续控制路径。
            raise ValueError("expected_length 必须是非负整数")  # 遇到非法合同立即显式失败。
        q = self._split(self.q_proj(x_t))  # 计算并保存当前步骤的中间状态。
        k_new, v_new = self._split(self.k_proj(x_t)), self._split(self.v_proj(x_t))  # 计算并保存当前步骤的中间状态。
        if cache is None:  # 按当前条件选择后续控制路径。
            if expected_length != 0:  # 按当前条件选择后续控制路径。
                raise ValueError("非零 position 不允许空 cache")  # 遇到非法合同立即显式失败。
            if not bool(valid_t.all()):  # 按当前条件选择后续控制路径。
                raise ValueError("每条增量序列的首 token 必须有效")  # 遇到非法合同立即显式失败。
            k_all, v_all, key_valid = k_new, v_new, valid_t  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            if expected_length == 0:  # 按当前条件选择后续控制路径。
                raise ValueError("position=0 不允许已有 cache")  # 遇到非法合同立即显式失败。
            k_old, v_old, valid_old = self._validate_cache(  # 计算并保存当前步骤的中间状态。
                cache, x_t.shape[0], expected_length, x_t  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
            if bool((valid_t[:, 0] & ~valid_old[:, -1]).any()):  # 按当前条件选择后续控制路径。
                raise ValueError("cache 有效前缀已结束，不能再次追加有效 token")  # 遇到非法合同立即显式失败。
            k_all = torch.cat([k_old, k_new], dim=2)  # 计算并保存当前步骤的中间状态。
            v_all = torch.cat([v_old, v_new], dim=2)  # 计算并保存当前步骤的中间状态。
            key_valid = torch.cat([valid_old, valid_t], dim=1)  # 计算并保存当前步骤的中间状态。
        allowed = key_valid[:, None, None, :]  # 计算并保存当前步骤的中间状态。
        context, _ = scaled_dot_product_attention(q, k_all, v_all, allowed)  # 计算并保存当前步骤的中间状态。
        y = self.out_proj(self._merge(context)) * valid_t.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return y, {"k": k_all, "v": v_all, "valid": key_valid}  # 返回当前分支计算出的结果。


class GPTBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, n_heads, mlp_ratio=4):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.ln1 = nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.attn = ManualCausalSelfAttention(d_model, n_heads)  # 计算并保存当前步骤的中间状态。
        self.ln2 = nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.mlp = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Linear(d_model, mlp_ratio * d_model), nn.GELU(),  # 执行当前语句以推进本节示例。
            nn.Linear(mlp_ratio * d_model, d_model),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。

    def forward(self, x, valid_mask):  # 定义本节可复用的核心函数。
        a, weights = self.attn(self.ln1(x), valid_mask)  # 计算并保存当前步骤的中间状态。
        x = (x + a) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        x = (x + self.mlp(self.ln2(x))) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return x, weights  # 返回当前分支计算出的结果。

    def forward_step(self, x_t, cache, valid_t, expected_length):  # 定义本节可复用的核心函数。
        a, new_cache = self.attn.forward_step(  # 计算并保存当前步骤的中间状态。
            self.ln1(x_t), cache, valid_t, expected_length  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        x_t = (x_t + a) * valid_t.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        x_t = (x_t + self.mlp(self.ln2(x_t))) * valid_t.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return x_t, new_cache  # 返回当前分支计算出的结果。


class ScratchGPT(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, max_len, d_model=32, n_heads=4, n_layers=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if vocab_size <= 4 or max_len <= 1 or n_layers <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("模型配置必须为正且词表需包含普通 token")  # 遇到非法合同立即显式失败。
        self.config = dict(vocab_size=vocab_size, max_len=max_len, d_model=d_model,  # 计算并保存当前步骤的中间状态。
                           n_heads=n_heads, n_layers=n_layers)  # 计算并保存当前步骤的中间状态。
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD)  # 计算并保存当前步骤的中间状态。
        self.position_embedding = nn.Embedding(max_len, d_model)  # 计算并保存当前步骤的中间状态。
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads) for _ in range(n_layers)])  # 计算并保存当前步骤的中间状态。
        self.final_norm = nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)  # 计算并保存当前步骤的中间状态。
        self.lm_head.weight = self.token_embedding.weight  # 计算并保存当前步骤的中间状态。

    def _check_ids(self, ids):  # 定义本节可复用的核心函数。
        if ids.dtype != torch.long or ids.ndim != 2:  # 按当前条件选择后续控制路径。
            raise ValueError("input_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
        if ids.numel() and (int(ids.min()) < 0 or int(ids.max()) >= self.config["vocab_size"]):  # 按当前条件选择后续控制路径。
            raise ValueError("token id 越界")  # 遇到非法合同立即显式失败。

    def forward(self, input_ids, valid_mask=None):  # 定义本节可复用的核心函数。
        self._check_ids(input_ids)  # 执行当前语句以推进本节示例。
        B, T = input_ids.shape  # 计算并保存当前步骤的中间状态。
        if T > self.config["max_len"]:  # 按当前条件选择后续控制路径。
            raise ValueError("序列超过 max_len")  # 遇到非法合同立即显式失败。
        valid_mask = input_ids.ne(PAD) if valid_mask is None else valid_mask  # 计算并保存当前步骤的中间状态。
        if valid_mask.shape != input_ids.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("mask 与 ids 形状不一致")  # 遇到非法合同立即显式失败。
        validate_prefix_mask(valid_mask)  # 执行当前语句以推进本节示例。
        pos = torch.arange(T, device=input_ids.device)  # 计算并保存当前步骤的中间状态。
        x = self.token_embedding(input_ids) + self.position_embedding(pos)[None]  # 计算并保存当前步骤的中间状态。
        x = x * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        all_weights = []  # 计算并保存当前步骤的中间状态。
        for block in self.blocks:  # 遍历输入元素以累积或检查结果。
            x, weights = block(x, valid_mask)  # 计算并保存当前步骤的中间状态。
            all_weights.append(weights)  # 执行当前语句以推进本节示例。
        x = self.final_norm(x) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return self.lm_head(x), all_weights  # 返回当前分支计算出的结果。

    def _validate_cache_envelope(self, caches, batch_size, position, request_id, valid_t, device):  # 定义本节可复用的核心函数。
        if caches is None:  # 按当前条件选择后续控制路径。
            if position != 0:  # 按当前条件选择后续控制路径。
                raise ValueError("空 cache 时 position 必须为 0")  # 遇到非法合同立即显式失败。
            if not bool(valid_t.all()):  # 按当前条件选择后续控制路径。
                raise ValueError("每条请求的首 token 必须有效")  # 遇到非法合同立即显式失败。
            return [None] * len(self.blocks), torch.ones(batch_size, dtype=torch.bool, device=device)  # 返回当前分支计算出的结果。
        required = {"schema", "request_id", "batch_size", "next_position", "prefix_open", "layers"}  # 计算并保存当前步骤的中间状态。
        if not isinstance(caches, dict) or set(caches) != required:  # 按当前条件选择后续控制路径。
            raise ValueError("cache envelope 字段不完整")  # 遇到非法合同立即显式失败。
        if caches["schema"] != "scratch-gpt-kv-v1":  # 按当前条件选择后续控制路径。
            raise ValueError("cache schema 不支持")  # 遇到非法合同立即显式失败。
        if caches["request_id"] != request_id:  # 按当前条件选择后续控制路径。
            raise ValueError("cache 不属于当前请求")  # 遇到非法合同立即显式失败。
        if caches["batch_size"] != batch_size:  # 按当前条件选择后续控制路径。
            raise ValueError("cache batch size 与当前请求不一致")  # 遇到非法合同立即显式失败。
        if caches["next_position"] != position:  # 按当前条件选择后续控制路径。
            raise ValueError("position 必须严格等于 cache length")  # 遇到非法合同立即显式失败。
        prefix_open = caches["prefix_open"]  # 计算并保存当前步骤的中间状态。
        if not isinstance(prefix_open, torch.Tensor) or prefix_open.shape != (batch_size,):  # 按当前条件选择后续控制路径。
            raise ValueError("cache prefix_open shape 错误")  # 遇到非法合同立即显式失败。
        if prefix_open.dtype != torch.bool or prefix_open.device != device:  # 按当前条件选择后续控制路径。
            raise ValueError("cache prefix_open dtype/device 错误")  # 遇到非法合同立即显式失败。
        layers = caches["layers"]  # 计算并保存当前步骤的中间状态。
        if not isinstance(layers, list) or len(layers) != len(self.blocks):  # 按当前条件选择后续控制路径。
            raise ValueError("cache 层数与模型不一致")  # 遇到非法合同立即显式失败。
        if bool((valid_t[:, 0] & ~prefix_open).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("有效前缀结束后不能再次追加普通 token")  # 遇到非法合同立即显式失败。
        return layers, prefix_open  # 返回当前分支计算出的结果。

    def forward_step(self, token_id, caches=None, position=0, valid_t=None, request_id=None):  # 定义本节可复用的核心函数。
        self._check_ids(token_id)  # 执行当前语句以推进本节示例。
        if token_id.shape[1] != 1:  # 按当前条件选择后续控制路径。
            raise ValueError("forward_step 一次只接收一个 token")  # 遇到非法合同立即显式失败。
        if not isinstance(position, int) or isinstance(position, bool):  # 按当前条件选择后续控制路径。
            raise ValueError("position 必须是整数")  # 遇到非法合同立即显式失败。
        if not 0 <= position < self.config["max_len"]:  # 按当前条件选择后续控制路径。
            raise ValueError("position 越界")  # 遇到非法合同立即显式失败。
        if not isinstance(request_id, str) or not request_id:  # 按当前条件选择后续控制路径。
            raise ValueError("request_id 必须是非空字符串")  # 遇到非法合同立即显式失败。
        B = token_id.shape[0]  # 计算并保存当前步骤的中间状态。
        valid_t = token_id.ne(PAD) if valid_t is None else valid_t  # 计算并保存当前步骤的中间状态。
        if valid_t.shape != (B, 1) or valid_t.dtype != torch.bool or valid_t.device != token_id.device:  # 按当前条件选择后续控制路径。
            raise ValueError("valid_t shape/dtype/device 合同错误")  # 遇到非法合同立即显式失败。
        if not torch.equal(valid_t, token_id.ne(PAD)):  # 按当前条件选择后续控制路径。
            raise ValueError("valid_t 必须与 token_id 的 PAD 语义一致")  # 遇到非法合同立即显式失败。
        layer_caches, prefix_open = self._validate_cache_envelope(  # 计算并保存当前步骤的中间状态。
            caches, B, position, request_id, valid_t, token_id.device  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        pos = torch.tensor([position], device=token_id.device)  # 计算并保存当前步骤的中间状态。
        x = (self.token_embedding(token_id) + self.position_embedding(pos)[None]) * valid_t.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        new_layers = []  # 计算并保存当前步骤的中间状态。
        for block, layer_cache in zip(self.blocks, layer_caches):  # 遍历输入元素以累积或检查结果。
            x, new_cache = block.forward_step(x, layer_cache, valid_t, expected_length=position)  # 计算并保存当前步骤的中间状态。
            new_layers.append(new_cache)  # 执行当前语句以推进本节示例。
        reference_valid = new_layers[0]["valid"]  # 计算并保存当前步骤的中间状态。
        if any(not torch.equal(layer["valid"], reference_valid) for layer in new_layers[1:]):  # 按当前条件选择后续控制路径。
            raise RuntimeError("各层 cache valid 状态不一致")  # 遇到非法合同立即显式失败。
        x = self.final_norm(x) * valid_t.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        new_envelope = {  # 计算并保存当前步骤的中间状态。
            "schema": "scratch-gpt-kv-v1",  # 执行当前语句以推进本节示例。
            "request_id": request_id,  # 执行当前语句以推进本节示例。
            "batch_size": B,  # 执行当前语句以推进本节示例。
            "next_position": position + 1,  # 执行当前语句以推进本节示例。
            "prefix_open": prefix_open & valid_t[:, 0],  # 执行当前语句以推进本节示例。
            "layers": new_layers,  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
        return self.lm_head(x), new_envelope  # 返回当前分支计算出的结果。

In [ ]:
gpt = ScratchGPT(len(TOKENS), max_len=12, d_model=24, n_heads=4, n_layers=2).to(DEVICE)  # 计算并保存当前步骤的中间状态。
sample_ids = torch.tensor([[BOS, 4, 5, EOS, PAD], [BOS, 8, 9, 10, EOS]])  # 计算并保存当前步骤的中间状态。
sample_mask = sample_ids.ne(PAD)  # 计算并保存当前步骤的中间状态。
sample_logits, sample_weights = gpt(sample_ids, sample_mask)  # 计算并保存当前步骤的中间状态。

assert sample_logits.shape == (2, 5, len(TOKENS))  # 用受控断言验证关键不变量。
assert len(sample_weights) == 2 and sample_weights[0].shape == (2, 4, 5, 5)  # 用受控断言验证关键不变量。
assert gpt.lm_head.weight is gpt.token_embedding.weight  # 用受控断言验证关键不变量。
assert torch.count_nonzero(sample_logits[0, 4]) == 0  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in gpt.parameters()) < 50_000  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    gpt(torch.tensor([[BOS, PAD, 4]]), torch.tensor([[True, False, True]]))  # 执行当前语句以推进本节示例。
    raise AssertionError("非前缀 mask 应被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "左对齐" in str(exc)  # 用受控断言验证关键不变量。

## 5. 三个关键 oracle：未来隔离、KV Cache 等价与 cache 状态机

1. **未来 token 干预**：只改位置 `j`，所有 `i<j` 的 logits 必须逐元素不变。
2. **缓存等价**：eval 模式从位置 0 逐步喂入 token，拼接出的 logits 应与一次性 full forward 相同；还必须覆盖同 batch 中不同有效长度。
3. **状态机 fail closed**：`position` 必须等于 cache 的下一位置；每层 K/V 必须与当前 batch、head、长度、dtype、device 对齐；request id 必须一致；某行一旦进入 padding，就不能再次追加普通 token。

cache 使用 envelope 保存请求、batch、下一位置、前缀是否仍开放和逐层 K/V。它不是可跨请求复制的普通字典。在长度为 `T` 的生成中，full 重算累计约为 $O(T^3D)$，KV Cache 约为 $O(T^2D)$，代价是每层 $O(BTD)$ cache 内存。

In [ ]:
gpt.eval()  # 执行当前语句以推进本节示例。
probe = torch.tensor([[BOS, 4, 5, 6, 7, EOS], [BOS, 8, 9, 10, 11, EOS]])  # 计算并保存当前步骤的中间状态。
probe_mask = torch.ones_like(probe, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
changed = probe.clone()  # 计算并保存当前步骤的中间状态。
changed[:, 4] = torch.tensor([12, 13])  # 计算并保存当前步骤的中间状态。

with torch.no_grad():  # 在受管理的上下文中执行操作。
    full_logits, _ = gpt(probe, probe_mask)  # 计算并保存当前步骤的中间状态。
    changed_logits, _ = gpt(changed, probe_mask)  # 计算并保存当前步骤的中间状态。
assert torch.equal(full_logits[:, :4], changed_logits[:, :4])  # 用受控断言验证关键不变量。
assert not torch.allclose(full_logits[:, 4], changed_logits[:, 4])  # 用受控断言验证关键不变量。

caches, step_logits = None, []  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for position in range(probe.shape[1]):  # 遍历输入元素以累积或检查结果。
        logits_t, caches = gpt.forward_step(  # 计算并保存当前步骤的中间状态。
            probe[:, position:position + 1], caches, position,  # 执行当前语句以推进本节示例。
            valid_t=probe_mask[:, position:position + 1], request_id="oracle-full"  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        step_logits.append(logits_t)  # 执行当前语句以推进本节示例。
cached_logits = torch.cat(step_logits, dim=1)  # 计算并保存当前步骤的中间状态。
max_cache_error = (cached_logits - full_logits).abs().max().item()  # 计算并保存当前步骤的中间状态。
assert torch.allclose(cached_logits, full_logits, atol=7e-6, rtol=1e-5), max_cache_error  # 用受控断言验证关键不变量。
assert caches["next_position"] == probe.shape[1]  # 用受控断言验证关键不变量。
assert all(layer["k"].shape[2] == probe.shape[1] for layer in caches["layers"])  # 用受控断言验证关键不变量。
assert all(layer["valid"].all() for layer in caches["layers"])  # 用受控断言验证关键不变量。

# 同一个 batch 中第一行有尾部 padding，full 与 cache 仍需一致。
padded = torch.tensor([[BOS, 4, 5, EOS, PAD, PAD], [BOS, 8, 9, 10, 11, EOS]])  # 计算并保存当前步骤的中间状态。
padded_mask = padded.ne(PAD)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    padded_full = gpt(padded, padded_mask)[0]  # 计算并保存当前步骤的中间状态。
    padded_cache, padded_steps = None, []  # 计算并保存当前步骤的中间状态。
    for position in range(padded.shape[1]):  # 遍历输入元素以累积或检查结果。
        logits_t, padded_cache = gpt.forward_step(  # 计算并保存当前步骤的中间状态。
            padded[:, position:position + 1], padded_cache, position,  # 执行当前语句以推进本节示例。
            valid_t=padded_mask[:, position:position + 1], request_id="oracle-padded"  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        padded_steps.append(logits_t)  # 执行当前语句以推进本节示例。
padded_cached = torch.cat(padded_steps, dim=1)  # 计算并保存当前步骤的中间状态。
padded_error = float((padded_cached - padded_full).abs().max())  # 计算并保存当前步骤的中间状态。
assert torch.allclose(padded_cached, padded_full, atol=7e-6, rtol=1e-5), padded_error  # 用受控断言验证关键不变量。
assert padded_cache["prefix_open"].tolist() == [False, True]  # 用受控断言验证关键不变量。

# position、request、shape 与有效前缀反例必须在接口层被拒绝。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _, one_step_cache = gpt.forward_step(  # 计算并保存当前步骤的中间状态。
        torch.tensor([[BOS]]), None, 0, request_id="request-a"  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。

def expect_cache_error(action, phrase):  # 定义本节可复用的核心函数。
    try:  # 尝试执行可能失败的受控操作。
        action()  # 执行当前语句以推进本节示例。
        raise AssertionError("非法 cache 必须 fail closed")  # 遇到非法合同立即显式失败。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        assert phrase in str(exc), str(exc)  # 用受控断言验证关键不变量。

expect_cache_error(  # 执行当前语句以推进本节示例。
    lambda: gpt.forward_step(torch.tensor([[4]]), one_step_cache, 3, request_id="request-a"),  # 计算并保存当前步骤的中间状态。
    "position",  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
expect_cache_error(  # 执行当前语句以推进本节示例。
    lambda: gpt.forward_step(torch.tensor([[4]]), one_step_cache, 1, request_id="request-b"),  # 计算并保存当前步骤的中间状态。
    "请求",  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
bad_shape_cache = copy.deepcopy(one_step_cache)  # 计算并保存当前步骤的中间状态。
bad_shape_cache["layers"][0]["k"] = bad_shape_cache["layers"][0]["k"][:, :-1]  # 计算并保存当前步骤的中间状态。
expect_cache_error(  # 执行当前语句以推进本节示例。
    lambda: gpt.forward_step(torch.tensor([[4]]), bad_shape_cache, 1, request_id="request-a"),  # 计算并保存当前步骤的中间状态。
    "shape",  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _, gap_cache = gpt.forward_step(torch.tensor([[BOS]]), None, 0, request_id="gap")  # 计算并保存当前步骤的中间状态。
    _, gap_cache = gpt.forward_step(torch.tensor([[PAD]]), gap_cache, 1, request_id="gap")  # 计算并保存当前步骤的中间状态。
expect_cache_error(  # 执行当前语句以推进本节示例。
    lambda: gpt.forward_step(torch.tensor([[4]]), gap_cache, 2, request_id="gap"),  # 计算并保存当前步骤的中间状态。
    "前缀",  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
expect_cache_error(  # 执行当前语句以推进本节示例。
    lambda: gpt.forward_step(  # 执行当前语句以推进本节示例。
        torch.tensor([[PAD]]), None, 0,  # 执行当前语句以推进本节示例。
        valid_t=torch.ones(1, 1, dtype=torch.bool), request_id="pad-semantic"  # 计算并保存当前步骤的中间状态。
    ),  # 执行当前语句以推进本节示例。
    "PAD",  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
print({"full_vs_cache_max_abs": max_cache_error,  # 执行当前语句以推进本节示例。
       "padded_full_vs_cache_max_abs": padded_error,  # 执行当前语句以推进本节示例。
       "cache_rejections": ["position", "request", "shape", "valid_hole", "pad_semantic"]})  # 执行当前语句以推进本节示例。

## 6. Causal LM 目标与受控过拟合

teacher forcing 的第 `t` 个 logits 预测第 `t+1` 个 token：

$$\mathcal L=-\frac{1}{N}\sum_{b,t:\,m_{b,t+1}=1}\log p(x_{b,t+1}\mid x_{b,\le t}).$$

padding 位置不是一个“PAD 类别预测任务”，因此监督标签改为 `ignore_index=-100`。下面同时以布尔选取手算一次 loss，防止 shift 或 mask 错一位。训练只要求记住固定合成序列，是实现冒烟测试，不是泛化基准。

In [ ]:
def make_toy_sequences():  # 定义本节可复用的核心函数。
    rows = []  # 计算并保存当前步骤的中间状态。
    for i in range(12):  # 遍历输入元素以累积或检查结果。
        length = 6 + (i % 3)  # 计算并保存当前步骤的中间状态。
        regular = [4 + ((i + j) % (len(TOKENS) - 4)) for j in range(length - 2)]  # 计算并保存当前步骤的中间状态。
        row = [BOS] + regular + [EOS]  # 计算并保存当前步骤的中间状态。
        rows.append(row + [PAD] * (9 - len(row)))  # 执行当前语句以推进本节示例。
    return torch.tensor(rows, dtype=torch.long)  # 返回当前分支计算出的结果。


def causal_lm_loss(logits, input_ids, valid_mask):  # 定义本节可复用的核心函数。
    if input_ids.ndim != 2 or input_ids.dtype != torch.long:  # 按当前条件选择后续控制路径。
        raise ValueError("input_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
    if logits.shape[:2] != input_ids.shape or valid_mask.shape != input_ids.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("loss 输入形状不一致")  # 遇到非法合同立即显式失败。
    if valid_mask.dtype != torch.bool or logits.shape[-1] <= int(input_ids.max()):  # 按当前条件选择后续控制路径。
        raise ValueError("loss mask dtype 或 vocab 维错误")  # 遇到非法合同立即显式失败。
    if not torch.equal(valid_mask, input_ids.ne(PAD)):  # 按当前条件选择后续控制路径。
        raise ValueError("loss valid_mask 必须与 PAD 位置一致")  # 遇到非法合同立即显式失败。
    validate_prefix_mask(valid_mask)  # 执行当前语句以推进本节示例。
    supervised = valid_mask[:, 1:]  # 计算并保存当前步骤的中间状态。
    if input_ids.shape[1] < 2 or not bool(supervised.any()):  # 按当前条件选择后续控制路径。
        raise ValueError("当前 batch 没有有效 next-token label")  # 遇到非法合同立即显式失败。
    targets = input_ids[:, 1:].clone()  # 计算并保存当前步骤的中间状态。
    targets[~supervised] = -100  # 计算并保存当前步骤的中间状态。
    return F.cross_entropy(  # 返回当前分支计算出的结果。
        logits[:, :-1].reshape(-1, logits.shape[-1]), targets.reshape(-1), ignore_index=-100  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。


toy_ids = make_toy_sequences().to(DEVICE)  # 计算并保存当前步骤的中间状态。
toy_mask = toy_ids.ne(PAD)  # 计算并保存当前步骤的中间状态。
test_logits, _ = gpt(toy_ids, toy_mask)  # 计算并保存当前步骤的中间状态。
loss_api = causal_lm_loss(test_logits, toy_ids, toy_mask)  # 计算并保存当前步骤的中间状态。
selected_logits = test_logits[:, :-1][toy_mask[:, 1:]]  # 计算并保存当前步骤的中间状态。
selected_targets = toy_ids[:, 1:][toy_mask[:, 1:]]  # 计算并保存当前步骤的中间状态。
loss_manual = F.cross_entropy(selected_logits, selected_targets)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(loss_api, loss_manual)  # 用受控断言验证关键不变量。
assert int(toy_mask[:, 1:].sum()) == selected_targets.numel()  # 用受控断言验证关键不变量。

single_token = torch.tensor([[BOS, PAD]])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    causal_lm_loss(gpt(single_token, single_token.ne(PAD))[0], single_token, single_token.ne(PAD))  # 执行当前语句以推进本节示例。
    raise AssertionError("全 ignore label 必须被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "next-token" in str(exc)  # 用受控断言验证关键不变量。

In [ ]:
torch.manual_seed(SEED + 1)  # 执行当前语句以推进本节示例。
gpt = ScratchGPT(len(TOKENS), max_len=12, d_model=24, n_heads=4, n_layers=2).to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.AdamW(gpt.parameters(), lr=0.025, weight_decay=0.0)  # 计算并保存当前步骤的中间状态。

gpt.train()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_loss = float(causal_lm_loss(gpt(toy_ids, toy_mask)[0], toy_ids, toy_mask))  # 计算并保存当前步骤的中间状态。
for step in range(100):  # 遍历输入元素以累积或检查结果。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    loss = causal_lm_loss(gpt(toy_ids, toy_mask)[0], toy_ids, toy_mask)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        grad_norm = torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)  # 计算并保存当前步骤的中间状态。
        assert torch.isfinite(grad_norm) and float(grad_norm) > 0  # 用受控断言验证关键不变量。
    else:  # 处理前置条件不成立的分支。
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。

gpt.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    final_loss = float(causal_lm_loss(gpt(toy_ids, toy_mask)[0], toy_ids, toy_mask))  # 计算并保存当前步骤的中间状态。
assert final_loss < initial_loss * 0.35, (initial_loss, final_loss)  # 用受控断言验证关键不变量。
assert torch.isfinite(torch.tensor(final_loss))  # 用受控断言验证关键不变量。
print({"initial_loss": round(initial_loss, 4), "final_loss": round(final_loss, 4)})  # 执行当前语句以推进本节示例。

## 7. top-k / temperature 增量生成

temperature 对 logits 做 $z/\tau$；top-k 只在最高的 k 个候选中采样。生成 prompt 必须是无 PAD 的有效前缀，负 token 预算直接拒绝。PAD 永远从候选中屏蔽，因此不会出现“先 PAD、后普通 token”的非法 cache；EOS 立即停止。

循环只在还需要下一次采样时，才把刚生成 token 送入 `forward_step`，避免生成完最后一个 token 后做一次无用 cache 更新。相同请求级 generator seed 必须得到相同序列；真实服务还需要停止词、敏感内容策略、重复惩罚和不可伪造的请求身份。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def generate(model, prompt, max_new_tokens, temperature=1.0, top_k=5, seed=0):  # 定义本节可复用的核心函数。
    if temperature <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("temperature 必须为正")  # 遇到非法合同立即显式失败。
    if not isinstance(max_new_tokens, int) or isinstance(max_new_tokens, bool) or max_new_tokens < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("max_new_tokens 必须是非负整数")  # 遇到非法合同立即显式失败。
    if not 1 <= top_k <= model.config["vocab_size"] - 1:  # 按当前条件选择后续控制路径。
        raise ValueError("top_k 越界；PAD 不属于候选")  # 遇到非法合同立即显式失败。
    if prompt.ndim != 2 or prompt.shape[0] != 1 or prompt.shape[1] == 0:  # 按当前条件选择后续控制路径。
        raise ValueError("示例生成器只接收非空单样本 [1,T]")  # 遇到非法合同立即显式失败。
    model._check_ids(prompt)  # 执行当前语句以推进本节示例。
    prompt_mask = prompt.ne(PAD)  # 计算并保存当前步骤的中间状态。
    validate_prefix_mask(prompt_mask)  # 执行当前语句以推进本节示例。
    if not bool(prompt_mask.all()):  # 按当前条件选择后续控制路径。
        raise ValueError("生成 prompt 不允许包含 PAD 或 mask hole")  # 遇到非法合同立即显式失败。
    if prompt.device != next(model.parameters()).device:  # 按当前条件选择后续控制路径。
        raise ValueError("prompt 与模型必须位于同一 device")  # 遇到非法合同立即显式失败。
    if prompt.shape[1] + max_new_tokens > model.config["max_len"]:  # 按当前条件选择后续控制路径。
        raise ValueError("生成长度超过位置表")  # 遇到非法合同立即显式失败。
    generated = prompt.clone()  # 计算并保存当前步骤的中间状态。
    if max_new_tokens == 0 or int(prompt[0, -1]) == EOS:  # 按当前条件选择后续控制路径。
        return generated  # 返回当前分支计算出的结果。
    model.eval()  # 执行当前语句以推进本节示例。
    request_id = f"generation:{int(seed)}:{prompt.shape[1]}"  # 计算并保存当前步骤的中间状态。
    caches, last_logits = None, None  # 计算并保存当前步骤的中间状态。
    for position in range(prompt.shape[1]):  # 遍历输入元素以累积或检查结果。
        last_logits, caches = model.forward_step(  # 计算并保存当前步骤的中间状态。
            prompt[:, position:position + 1], caches, position,  # 执行当前语句以推进本节示例。
            valid_t=torch.ones(1, 1, dtype=torch.bool, device=prompt.device),  # 计算并保存当前步骤的中间状态。
            request_id=request_id,  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
    rng = torch.Generator(device=prompt.device).manual_seed(int(seed))  # 计算并保存当前步骤的中间状态。
    for step in range(max_new_tokens):  # 遍历输入元素以累积或检查结果。
        scores = (last_logits[:, -1] / temperature).clone()  # 计算并保存当前步骤的中间状态。
        scores[:, PAD] = float("-inf")  # 计算并保存当前步骤的中间状态。
        top_values, top_indices = torch.topk(scores, top_k, dim=-1)  # 计算并保存当前步骤的中间状态。
        probs = torch.softmax(top_values, dim=-1)  # 计算并保存当前步骤的中间状态。
        choice = torch.multinomial(probs, 1, generator=rng)  # 计算并保存当前步骤的中间状态。
        next_id = top_indices.gather(-1, choice)  # 计算并保存当前步骤的中间状态。
        if int(next_id.item()) == PAD:  # 按当前条件选择后续控制路径。
            raise RuntimeError("PAD 已从生成候选中排除")  # 遇到非法合同立即显式失败。
        generated = torch.cat([generated, next_id], dim=1)  # 计算并保存当前步骤的中间状态。
        if int(next_id.item()) == EOS or step == max_new_tokens - 1:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        position = prompt.shape[1] + step  # 计算并保存当前步骤的中间状态。
        last_logits, caches = model.forward_step(  # 计算并保存当前步骤的中间状态。
            next_id, caches, position,  # 执行当前语句以推进本节示例。
            valid_t=torch.ones_like(next_id, dtype=torch.bool), request_id=request_id  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
    return generated  # 返回当前分支计算出的结果。


prompt = torch.tensor([[BOS, 4]])  # 计算并保存当前步骤的中间状态。
generated_a = generate(gpt, prompt, 5, temperature=0.8, top_k=4, seed=77)  # 计算并保存当前步骤的中间状态。
generated_b = generate(gpt, prompt, 5, temperature=0.8, top_k=4, seed=77)  # 计算并保存当前步骤的中间状态。
assert torch.equal(generated_a, generated_b)  # 用受控断言验证关键不变量。
assert torch.equal(generated_a[:, :2], prompt)  # 用受控断言验证关键不变量。
assert generated_a.shape[1] <= 7 and not bool(generated_a.eq(PAD).any())  # 用受控断言验证关键不变量。
assert torch.equal(generate(gpt, prompt, 0), prompt)  # 用受控断言验证关键不变量。

for invalid_prompt, budget, phrase in [  # 遍历输入元素以累积或检查结果。
    (prompt, -1, "非负"),  # 执行当前语句以推进本节示例。
    (torch.tensor([[BOS, PAD, 4]]), 1, "前缀"),  # 执行当前语句以推进本节示例。
    (torch.tensor([[BOS, 4, PAD]]), 1, "PAD"),  # 执行当前语句以推进本节示例。
]:  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        generate(gpt, invalid_prompt, budget, seed=1)  # 计算并保存当前步骤的中间状态。
        raise AssertionError("非法生成请求必须被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        assert phrase in str(exc), str(exc)  # 用受控断言验证关键不变量。
print({"ids": generated_a.tolist()[0], "text": tokenizer.decode(generated_a.tolist()[0]),  # 执行当前语句以推进本节示例。
       "generation_rejections": ["negative_budget", "mask_hole", "trailing_pad"]})  # 执行当前语句以推进本节示例。

## 8. 发布者 registry 才是信任锚

package 内部的 SHA-256 只能发现偶然损坏：攻击者能同时替换模型、manifest 和内部哈希。这里先由“发布者侧”把 `(artifact_id, version) -> immutable manifest digest` 放入只读 registry；loader 必须先命中该外部锚，再验证原始 state bytes，最后按 tensor key、dtype、shape、bytes 重算语义权重哈希。

manifest 原子绑定完整 tokenizer token 顺序、无分类 label 的显式声明、训练数据与 split、预处理和训练 recipe。Notebook 中的 `MappingProxyType` 只是离线模拟；生产 registry 应位于受权限控制的数据库或签名透明日志，调用方不能写。

In [ ]:
from types import MappingProxyType  # 导入本单元所需的依赖。

def canonical_hash(obj):  # 定义本节可复用的核心函数。
    raw = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw).hexdigest()  # 返回当前分支计算出的结果。


def tensor_state_hash(state_dict):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state_dict):  # 遍历输入元素以累积或检查结果。
        tensor = state_dict[key]  # 计算并保存当前步骤的中间状态。
        if not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise TypeError("state_dict 只能包含 tensor")  # 遇到非法合同立即显式失败。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        descriptor = {"key": key, "dtype": str(value.dtype), "shape": list(value.shape)}  # 计算并保存当前步骤的中间状态。
        digest.update(json.dumps(descriptor, sort_keys=True, separators=(",", ":")).encode())  # 计算并保存当前步骤的中间状态。
        digest.update(value.numpy().tobytes(order="C"))  # 计算并保存当前步骤的中间状态。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def serialize_state(state_dict):  # 定义本节可复用的核心函数。
    buffer = io.BytesIO()  # 计算并保存当前步骤的中间状态。
    torch.save(state_dict, buffer)  # 执行当前语句以推进本节示例。
    return buffer.getvalue()  # 返回当前分支计算出的结果。


def validate_gpt_manifest(manifest):  # 定义本节可复用的核心函数。
    required = {"schema", "artifact_id", "version", "subject", "config", "tokenizer",  # 计算并保存当前步骤的中间状态。
                "label_map", "preprocess", "training_snapshot", "state_bytes_sha256",  # 执行当前语句以推进本节示例。
                "state_tensor_sha256"}  # 执行当前语句以推进本节示例。
    if not isinstance(manifest, dict) or set(manifest) != required:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest schema 字段不完整")  # 遇到非法合同立即显式失败。
    if manifest["schema"] != "scratch-gpt-v2" or not manifest["artifact_id"] or not manifest["version"]:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest 身份字段错误")  # 遇到非法合同立即显式失败。
    config = manifest["config"]  # 计算并保存当前步骤的中间状态。
    if set(config) != {"vocab_size", "max_len", "d_model", "n_heads", "n_layers"}:  # 按当前条件选择后续控制路径。
        raise ValueError("GPT config 字段错误")  # 遇到非法合同立即显式失败。
    tok_spec = manifest["tokenizer"]  # 计算并保存当前步骤的中间状态。
    if set(tok_spec) != {"kind", "tokens"} or tok_spec["kind"] != "whitespace-v1":  # 按当前条件选择后续控制路径。
        raise ValueError("tokenizer spec 错误")  # 遇到非法合同立即显式失败。
    tok = TinyTokenizer(tok_spec["tokens"])  # 计算并保存当前步骤的中间状态。
    if config["vocab_size"] != len(tok.tokens):  # 按当前条件选择后续控制路径。
        raise ValueError("vocab_size 与 tokenizer token 数不一致")  # 遇到非法合同立即显式失败。
    if manifest["label_map"] is not None:  # 按当前条件选择后续控制路径。
        raise ValueError("causal LM 不应伪造分类 label map")  # 遇到非法合同立即显式失败。
    preprocess = manifest["preprocess"]  # 计算并保存当前步骤的中间状态。
    expected_preprocess = {"splitter": "whitespace-v1", "padding": "right",  # 计算并保存当前步骤的中间状态。
                           "pad_id": PAD, "bos_id": BOS, "eos_id": EOS,  # 执行当前语句以推进本节示例。
                           "unk_id": UNK, "max_len": config["max_len"]}  # 执行当前语句以推进本节示例。
    if preprocess != expected_preprocess:  # 按当前条件选择后续控制路径。
        raise ValueError("预处理快照与模型合同不一致")  # 遇到非法合同立即显式失败。
    snapshot = manifest["training_snapshot"]  # 计算并保存当前步骤的中间状态。
    if snapshot.get("tokenizer_sha256") != canonical_hash(tok_spec):  # 按当前条件选择后续控制路径。
        raise ValueError("训练快照 tokenizer 指纹不一致")  # 遇到非法合同立即显式失败。
    dataset = snapshot.get("dataset", {})  # 计算并保存当前步骤的中间状态。
    rows, split = dataset.get("rows"), dataset.get("split")  # 计算并保存当前步骤的中间状态。
    if not isinstance(rows, list) or not rows or set(split or {}) != {"train", "validation", "test"}:  # 按当前条件选择后续控制路径。
        raise ValueError("训练数据或 split 快照错误")  # 遇到非法合同立即显式失败。
    all_indices = split["train"] + split["validation"] + split["test"]  # 计算并保存当前步骤的中间状态。
    if len(all_indices) != len(set(all_indices)) or sorted(all_indices) != list(range(len(rows))):  # 按当前条件选择后续控制路径。
        raise ValueError("训练 split 必须互斥且覆盖数据快照")  # 遇到非法合同立即显式失败。
    for row in rows:  # 遍历输入元素以累积或检查结果。
        if not isinstance(row, list) or not 2 <= len(row) <= config["max_len"]:  # 按当前条件选择后续控制路径。
            raise ValueError("训练行长度错误")  # 遇到非法合同立即显式失败。
        if any(not isinstance(token, int) or token < 0 or token >= config["vocab_size"] for token in row):  # 按当前条件选择后续控制路径。
            raise ValueError("训练 token id 越界")  # 遇到非法合同立即显式失败。
        valid = [token != PAD for token in row]  # 计算并保存当前步骤的中间状态。
        if any(valid[i] and not valid[i - 1] for i in range(1, len(valid))):  # 按当前条件选择后续控制路径。
            raise ValueError("训练数据包含 padding hole")  # 遇到非法合同立即显式失败。
    recipe = snapshot.get("recipe", {})  # 计算并保存当前步骤的中间状态。
    expected_recipe = {"objective": "causal-next-token", "optimizer": "AdamW",  # 计算并保存当前步骤的中间状态。
                       "steps": 100, "lr": 0.025, "weight_decay": 0.0,  # 执行当前语句以推进本节示例。
                       "clip_grad_norm": 1.0, "seed": SEED + 1,  # 执行当前语句以推进本节示例。
                       "ignore_index": -100, "purpose": "controlled-overfit"}  # 执行当前语句以推进本节示例。
    if recipe != expected_recipe:  # 按当前条件选择后续控制路径。
        raise ValueError("训练 recipe 快照不一致")  # 遇到非法合同立即显式失败。
    return tok  # 返回当前分支计算出的结果。


def build_artifact(model, tok, subject, artifact_id, version, training_snapshot):  # 定义本节可复用的核心函数。
    state_dict = model.state_dict()  # 计算并保存当前步骤的中间状态。
    state_bytes = serialize_state(state_dict)  # 计算并保存当前步骤的中间状态。
    manifest = {  # 计算并保存当前步骤的中间状态。
        "schema": "scratch-gpt-v2", "artifact_id": artifact_id, "version": version,  # 执行当前语句以推进本节示例。
        "subject": subject, "config": copy.deepcopy(model.config),  # 执行当前语句以推进本节示例。
        "tokenizer": copy.deepcopy(tok.spec()), "label_map": None,  # 执行当前语句以推进本节示例。
        "preprocess": {"splitter": "whitespace-v1", "padding": "right", "pad_id": PAD,  # 执行当前语句以推进本节示例。
                       "bos_id": BOS, "eos_id": EOS, "unk_id": UNK,  # 执行当前语句以推进本节示例。
                       "max_len": model.config["max_len"]},  # 执行当前语句以推进本节示例。
        "training_snapshot": copy.deepcopy(training_snapshot),  # 执行当前语句以推进本节示例。
        "state_bytes_sha256": hashlib.sha256(state_bytes).hexdigest(),  # 执行当前语句以推进本节示例。
        "state_tensor_sha256": tensor_state_hash(state_dict),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    validate_gpt_manifest(manifest)  # 执行当前语句以推进本节示例。
    return {"manifest": manifest, "manifest_sha256": canonical_hash(manifest),  # 返回当前分支计算出的结果。
            "state_bytes": state_bytes}  # 执行当前语句以推进本节示例。


training_snapshot31 = {  # 计算并保存当前步骤的中间状态。
    "dataset": {"name": "toy-causal-sequences-v1", "rows": toy_ids.cpu().tolist(),  # 执行当前语句以推进本节示例。
                "split": {"train": list(range(len(toy_ids))), "validation": [], "test": []}},  # 执行当前语句以推进本节示例。
    "tokenizer_sha256": canonical_hash(tokenizer.spec()),  # 执行当前语句以推进本节示例。
    "recipe": {"objective": "causal-next-token", "optimizer": "AdamW", "steps": 100,  # 执行当前语句以推进本节示例。
               "lr": 0.025, "weight_decay": 0.0, "clip_grad_norm": 1.0,  # 执行当前语句以推进本节示例。
               "seed": SEED + 1, "ignore_index": -100, "purpose": "controlled-overfit"},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact = build_artifact(  # 计算并保存当前步骤的中间状态。
    gpt, tokenizer, subject="nlp-lab/gpt-demo", artifact_id="scratch-gpt-demo", version="1.0.0",  # 计算并保存当前步骤的中间状态。
    training_snapshot=training_snapshot31,  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
# 这一步模拟发布系统写入的外部、只读信任锚；请求方只能提交 package，不能改 registry。
PUBLISHER_REGISTRY31 = MappingProxyType({  # 计算并保存当前步骤的中间状态。
    ("scratch-gpt-demo", "1.0.0"): artifact["manifest_sha256"]  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。


def trusted_load(package, expected_subject):  # 定义本节可复用的核心函数。
    if not isinstance(package, dict) or set(package) != {"manifest", "manifest_sha256", "state_bytes"}:  # 按当前条件选择后续控制路径。
        raise ValueError("package 字段错误")  # 遇到非法合同立即显式失败。
    manifest = package["manifest"]  # 计算并保存当前步骤的中间状态。
    key = (manifest.get("artifact_id"), manifest.get("version"))  # 计算并保存当前步骤的中间状态。
    expected_digest = PUBLISHER_REGISTRY31.get(key)  # 计算并保存当前步骤的中间状态。
    if expected_digest is None:  # 按当前条件选择后续控制路径。
        raise PermissionError("artifact id/version 未由发布者注册")  # 遇到非法合同立即显式失败。
    computed_manifest_digest = canonical_hash(manifest)  # 计算并保存当前步骤的中间状态。
    if package["manifest_sha256"] != computed_manifest_digest:  # 按当前条件选择后续控制路径。
        raise ValueError("package 内 manifest hash 不一致")  # 遇到非法合同立即显式失败。
    if computed_manifest_digest != expected_digest:  # 按当前条件选择后续控制路径。
        raise PermissionError("package 内容不匹配发布者 registry")  # 遇到非法合同立即显式失败。
    if manifest.get("subject") != expected_subject:  # 按当前条件选择后续控制路径。
        raise PermissionError("业务主体不匹配")  # 遇到非法合同立即显式失败。
    if hashlib.sha256(package["state_bytes"]).hexdigest() != manifest["state_bytes_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("原始权重字节指纹不匹配")  # 遇到非法合同立即显式失败。
    tok = validate_gpt_manifest(manifest)  # 计算并保存当前步骤的中间状态。
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)  # 计算并保存当前步骤的中间状态。
    if tensor_state_hash(state) != manifest["state_tensor_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("tensor key/dtype/shape/bytes 指纹不匹配")  # 遇到非法合同立即显式失败。
    model = ScratchGPT(**manifest["config"])  # 计算并保存当前步骤的中间状态。
    model.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return model.eval(), tok  # 返回当前分支计算出的结果。


loaded_gpt, loaded_tokenizer = trusted_load(artifact, "nlp-lab/gpt-demo")  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    original = gpt(probe, probe_mask)[0]  # 计算并保存当前步骤的中间状态。
    restored = loaded_gpt(probe, probe_mask)[0]  # 计算并保存当前步骤的中间状态。
assert torch.equal(original, restored)  # 用受控断言验证关键不变量。
assert loaded_tokenizer.tokens == tokenizer.tokens  # 用受控断言验证关键不变量。
assert gpt.lm_head.weight is gpt.token_embedding.weight  # 用受控断言验证关键不变量。

# 仅修改内容并重算 package 内 hash，仍不能改变发布者 registry 中的预期 digest。
resigned_config = copy.deepcopy(artifact)  # 计算并保存当前步骤的中间状态。
resigned_config["manifest"]["config"]["max_len"] += 1  # 计算并保存当前步骤的中间状态。
resigned_config["manifest_sha256"] = canonical_hash(resigned_config["manifest"])  # 计算并保存当前步骤的中间状态。
forged_model = ScratchGPT(**gpt.config)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for parameter in forged_model.parameters():  # 遍历输入元素以累积或检查结果。
        parameter.zero_()  # 执行当前语句以推进本节示例。
fully_resigned = build_artifact(  # 计算并保存当前步骤的中间状态。
    forged_model, tokenizer, "nlp-lab/gpt-demo", "scratch-gpt-demo", "1.0.0", training_snapshot31  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
for candidate in (resigned_config, fully_resigned):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        trusted_load(candidate, "nlp-lab/gpt-demo")  # 执行当前语句以推进本节示例。
        raise AssertionError("整体重签伪造必须被 registry 拒绝")  # 遇到非法合同立即显式失败。
    except PermissionError as exc:  # 捕获预期异常并验证失败分支。
        assert "registry" in str(exc)  # 用受控断言验证关键不变量。

mismatched_vocab_model = ScratchGPT(len(TOKENS) + 1, max_len=12, d_model=24, n_heads=4, n_layers=2)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    build_artifact(  # 执行当前语句以推进本节示例。
        mismatched_vocab_model, tokenizer, "nlp-lab/gpt-demo",  # 执行当前语句以推进本节示例。
        "mismatch", "1.0.0", training_snapshot31  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    raise AssertionError("vocab_size/tokenizer mismatch 必须在发布前拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "vocab_size" in str(exc)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    trusted_load(artifact, "another-tenant")  # 执行当前语句以推进本节示例。
    raise AssertionError("跨主体加载不应通过")  # 遇到非法合同立即显式失败。
except PermissionError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
print({"artifact_registry": dict(PUBLISHER_REGISTRY31),  # 执行当前语句以推进本节示例。
       "tensor_state_sha256_prefix": artifact["manifest"]["state_tensor_sha256"][:12],  # 执行当前语句以推进本节示例。
       "resigned_forgery_rejected": True, "snapshot_rows": len(training_snapshot31["dataset"]["rows"])})  # 执行当前语句以推进本节示例。

## 9. 复杂度、失败模式与生产边界

- 单层 full self-attention 时间/注意力矩阵空间分别约为 $O(BT^2D)$ / $O(BHT^2)$；MLP 时间约为 $O(BTD^2)$。
- KV Cache 降低重复计算，却随 batch、层数、上下文长度线性占显存；生产需分页 cache、配额、淘汰和请求隔离。
- 常见错误：cache 的位置 embedding 从 0 重启；只缓存最后一层；把 `True` 同时解释为“允许”和“屏蔽”；在训练态比较 dropout；权重共享发生在创建 optimizer 之后。
- padding 不是普通词。若右侧 PAD 参与 loss 或成为可见 key，模型会学习数据管道伪特征。
- 本实现没有 RoPE、FlashAttention、混合精度、分布式训练和连续批处理；这些是部署扩展，不应混进最小正确性 oracle。
- SHA-256 只证明字节未变，不能证明发布者可信；真实性需要签名和可信密钥。

## 10. 原始论文与官方资料

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762)：缩放点积、多头注意力与位置表示。
- Radford et al., [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)：decoder-style 生成式预训练。
- PyTorch 官方文档：[Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)、[Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)、[cross_entropy](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)。

复现论文时应记录数据版本、tokenizer、超参数和评估脚本；本笔记引用结构思想，不声称复现论文规模或指标。